In [ ]:
import os

import time
import numpy as np
import matplotlib.pyplot as plt

# os.environ["OMP_NUM_THREADS"] = "1"
# os.environ["MKL_NUM_THREADS"] = "1"
# os.environ["OPENBLAS_NUM_THREADS"] = "1"
# os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
# os.environ["NUMEXPR_NUM_THREADS"] = "1"


In [ ]:
def generate_gaussian_matrix(m: int, n: int, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    return rng.standard_normal((m, n))

In [ ]:
def generate_low_rank_matrix(m: int, n: int, true_rank: int = 20, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)

    U_r, _ = np.linalg.qr(rng.standard_normal((m, true_rank)))
    V_r, _ = np.linalg.qr(rng.standard_normal((n, true_rank)))

    singular_values = np.exp(-np.arange(true_rank) / 5.0)

    A = (U_r * singular_values) @ V_r.T

    noise_scale = 0.01 / np.sqrt(m * n)
    A += noise_scale * rng.standard_normal((m, n))
    return A

In [ ]:
def deterministic_svd(A: np.ndarray, k: int):
    m, n = A.shape
    
    gram = A.T @ A
    
    eigenvalues, V = np.linalg.eigh(gram)
    
    idx = np.argsort(eigenvalues)[::-1]
    s_sq = eigenvalues[idx][:k]
    V_k = V[:, idx][:, :k]
    
    s_k = np.sqrt(np.maximum(s_sq, 0))
    
    U_k = A @ V_k / s_k
    
    Vt_k = V_k.T
    
    return U_k, s_k, Vt_k

In [ ]:
def randomized_svd(A: np.ndarray, k: int, p: int = 10, seed: int = 0):
    m, n = A.shape
    l = k + p

    rng = np.random.default_rng(seed)
    Omega = rng.standard_normal((n, l))

    Y = A @ Omega
    Q, _ = np.linalg.qr(Y)

    B = Q.T @ A

    U_tilde, s, Vt = np.linalg.svd(B, full_matrices=False)
    U = Q @ U_tilde

    return U[:, :k], s[:k], Vt[:k, :]

In [ ]:
def frobenius_error(A: np.ndarray, U: np.ndarray, s: np.ndarray, Vt: np.ndarray) -> float:
    A_approx = (U * s) @ Vt
    return np.linalg.norm(A - A_approx, 'fro') / np.linalg.norm(A, 'fro')


In [ ]:
def explained_variance(A: np.ndarray, s: np.ndarray) -> float:
    variance_approx = np.sum(s**2)
    variance_total = np.sum(A**2)
    return float(variance_approx / variance_total)

In [ ]:
def timed_svd(method, *args, repeats: int = 3, **kwargs):
    times = []
    result = None
    for _ in range(repeats):
        t0 = time.perf_counter()
        result = method(*args, **kwargs)
        times.append(time.perf_counter() - t0)
    return result, float(np.median(times))

In [ ]:
def run_comparison(
    matrix_name: str,
    A: np.ndarray,
    k: int,
    p: int,
    repeats: int = 3,
):
    (U_d, s_d, Vt_d), t_det = timed_svd(deterministic_svd, A, k, repeats=repeats)
    err_det = frobenius_error(A, U_d, s_d, Vt_d)
    var_det = explained_variance(A, s_d)

    (U_r, s_r, Vt_r), t_rnd = timed_svd(randomized_svd, A, k, p, repeats=repeats)
    err_rnd = frobenius_error(A, U_r, s_r, Vt_r)
    var_rnd = explained_variance(A, s_r)

    return {
        "matrix":    matrix_name,
        "shape":     A.shape,
        "k":         k,
        "p":         p,
        "t_det":     t_det,
        "t_rnd":     t_rnd,
        "speedup":   t_det / t_rnd if t_rnd > 0 else float("inf"),
        "err_det":   err_det,
        "err_rnd":   err_rnd,
        "var_det":   var_det,
        "var_rnd":   var_rnd,
    }


In [ ]:
def print_results_table(results: list[dict]):
    """Pretty-print a comparison table."""

    col_w = {
        "matrix":   16,
        "shape":    14,
        "k":         5,
        "p":         5,
        "t_det":    10,
        "t_rnd":    10,
        "speedup":   9,
        "err_det":  10,
        "err_rnd":  10,
        "var_det":  10,
        "var_rnd":  10,
    }

    header = (
        f"{'Matrix':<{col_w['matrix']}}"
        f"{'Shape':<{col_w['shape']}}"
        f"{'k':>{col_w['k']}}"
        f"{'p':>{col_w['p']}}"
        f"{'Det(s)':>{col_w['t_det']}}"
        f"{'Rnd(s)':>{col_w['t_rnd']}}"
        f"{'Speedup':>{col_w['speedup']}}"
        f"{'Det Err':>{col_w['err_det']}}"
        f"{'Rnd Err':>{col_w['err_rnd']}}"
        f"{'Det Var':>{col_w['var_det']}}"
        f"{'Rnd Var':>{col_w['var_rnd']}}"
    )
    sep = "─" * len(header)

    print()
    print("╔" + "═" * len(header) + "╗")
    print("║" + " SVD Comparison: Deterministic vs. Randomized ".center(len(header)) + "║")
    print("╠" + "═" * len(header) + "╣")
    print("║" + header + "║")
    print("║" + sep + "║")

    for r in results:
        shape_str = f"{r['shape'][0]}×{r['shape'][1]}"
        row = (
            f"{r['matrix']:<{col_w['matrix']}}"
            f"{shape_str:<{col_w['shape']}}"
            f"{r['k']:>{col_w['k']}}"
            f"{r['p']:>{col_w['p']}}"
            f"{r['t_det']:>{col_w['t_det']}.4f}"
            f"{r['t_rnd']:>{col_w['t_rnd']}.4f}"
            f"{r['speedup']:>{col_w['speedup']}.2f}×"
            f"{r['err_det']:>{col_w['err_det']}.6f}"
            f"{r['err_rnd']:>{col_w['err_rnd']}.6f}"
            f"{r['var_det']:>{col_w['var_det']}.6f}"
            f"{r['var_rnd']:>{col_w['var_rnd']}.6f}"
        )
        print("║" + row + "║")

    print("╚" + "═" * len(header) + "╝")
    print()


In [ ]:
configs = [

        (200,  150,  10, 10),
        (200,  150,  20, 10),

        (800,  600,  20, 10),
        (800,  600,  40, 10),

        (1000, 1000, 50, 10),
        (2000, 1500, 50, 15),

        (3000, 2500, 100, 20),
        
        (10000, 3000, 100, 20),

    ]

results = []
repeats = 3   # median over this many runs to reduce timing noise

print(f"Running {len(configs) * 2} experiments "
      f"({repeats} timing repeats each)...\n")

for m, n, k, p in configs:
    # ── Gaussian (unstructured) ──────────────────────────────────────────
    A_gauss = generate_gaussian_matrix(m, n)
    res = run_comparison("Gaussian", A_gauss, k=k, p=p, repeats=repeats)
    results.append(res)
    print(f"  ✓  Gaussian  {m}×{n}  k={k}  p={p}")

    # ── Low-rank / spiky (structured) ────────────────────────────────────
    A_lowrank = generate_low_rank_matrix(m, n, true_rank=k)
    res = run_comparison("LowRank", A_lowrank, k=k, p=p, repeats=repeats)
    results.append(res)
    print(f"  ✓  LowRank   {m}×{n}  k={k}  p={p}")

print_results_table(results)


for m_type in ["Gaussian", "LowRank"]:
    type_results = [r for r in results if r["matrix"] == m_type]
    if not type_results:
        continue
    
    sizes = [r["shape"][0] for r in type_results]
    t_det = [r["t_det"] for r in type_results]
    t_rnd = [r["t_rnd"] for r in type_results]
    err_det = [r["err_det"] for r in type_results]
    err_rnd = [r["err_rnd"] for r in type_results]
    var_det = [r["var_det"] for r in type_results]
    var_rnd = [r["var_rnd"] for r in type_results]

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"SVD Performance Comparison: {m_type} Matrices")

    # Time plot
    ax1.plot(sizes, t_det, 'o-', label="Deterministic Time")
    ax1.plot(sizes, t_rnd, 's-', label="Randomized Time")
    ax1.set_xscale("log")
    ax1.set_yscale("log")
    ax1.set_xlabel("Matrix Size (N)")
    ax1.set_ylabel("Execution Time (seconds)")
    ax1.set_title("Execution Time vs Matrix Size")
    ax1.legend()
    ax1.grid(True, which="both", ls="--", alpha=0.5)

    # Error plot
    ax2.plot(sizes, err_det, 'o-', label="Deterministic Error")
    ax2.plot(sizes, err_rnd, 's-', label="Randomized Error")
    ax2.set_xscale("log")
    ax2.set_yscale("log")
    ax2.set_xlabel("Matrix Size (N)")
    ax2.set_ylabel("Relative Frobenius Error")
    ax2.set_title("Reconstruction Error vs Matrix Size")
    ax2.legend()
    ax2.grid(True, which="both", ls="--", alpha=0.5)

    # Variance plot
    ax3.plot(sizes, var_det, 'o-', label="Deterministic Var")
    ax3.plot(sizes, var_rnd, 's-', label="Randomized Var")
    ax3.set_xscale("log")
    ax3.set_xlabel("Matrix Size (N)")
    ax3.set_ylabel("Explained Variance Fraction")
    ax3.set_title("Explained Variance vs Matrix Size")
    ax3.legend()
    ax3.grid(True, which="both", ls="--", alpha=0.5)

    plt.tight_layout()
    filename = f"svd_comparison_{m_type.lower()}.png"
    plt.savefig(filename)
    print(f"Saved plot to {filename}")